# 进阶实践项目参考答案 04：脑疾病表格预测：校准、阈值与验证设计

从一次四分类结果继续检查验证策略、概率校准和模型解释的稳定性。

Kaggle 中先复制到自己的账户，再按任务顺序完成。题目只保留关键填写位置，数据读取、绘图和保存框架已经给出。

## 任务
1. 完成分层交叉验证
2. 比较逻辑回归与树模型
3. 校准类别概率
4. 统计高置信度错误
5. 比较变量重要性稳定性

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score,log_loss
from sklearn.calibration import calibration_curve
SEED=42; OUT=Path('advanced04_results'); OUT.mkdir(exist_ok=True); paths=list(Path('/kaggle/input').rglob('Data.csv'))+list(Path('.').rglob('Data.csv')); assert paths,'需要课程 Data.csv'
df=pd.read_csv(paths[0]); enc=LabelEncoder(); y=enc.fit_transform(df.label.astype(str)); X=df.drop(columns=['ID','label']); num=list(X.select_dtypes(include=np.number)); cat=[c for c in X if c not in num]
pre=ColumnTransformer([('num',make_pipeline(SimpleImputer(strategy='median'),StandardScaler()),num),('cat',make_pipeline(SimpleImputer(strategy='most_frequent'),OneHotEncoder(handle_unknown='ignore')),cat)])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED); model=make_pipeline(pre,LogisticRegression(max_iter=3000,class_weight='balanced')).fit(Xtr,ytr); prob=model.predict_proba(Xte); pred=prob.argmax(1); conf=prob.max(1); wrong=(pred!=yte)
top_wrong=int(((wrong)&(conf>=.8)).sum()); result={'macro_f1':float(f1_score(yte,pred,average='macro')),'log_loss':float(log_loss(yte,prob)),'high_confidence_errors':top_wrong,'test_size':len(yte)}
fig,ax=plt.subplots(1,2,figsize=(9,3.5)); ax[0].hist(conf[~wrong],bins=10,alpha=.6,label='correct'); ax[0].hist(conf[wrong],bins=10,alpha=.6,label='wrong'); ax[0].legend(); ax[0].set_title('Confidence by outcome'); ax[1].bar(enc.classes_,np.bincount(pred,minlength=len(enc.classes_))); ax[1].set_title('Predicted classes'); fig.tight_layout(); fig.savefig(OUT/'advanced04_summary.png',dpi=150); plt.close(fig); (OUT/'advanced04_result.json').write_text(json.dumps(result,indent=2),encoding='utf-8'); print(result)


{'macro_f1': 0.7450705360590211, 'log_loss': 0.6121723011531753, 'high_confidence_errors': 40, 'test_size': 750}
